In [ ]:
# Import and Configs

import os, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision

# Set Seed
SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

REAL_DIR = "/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train/real"
FAKE_DIR = "/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train/fake"

N_TRAIN = 5000   # per class
N_VAL   = 1000   # per class
N_TEST  = 1000   # per class

BATCH_SIZE  = 32  # per replica; effective = BATCH_SIZE × num_gpus
EPOCHS      = 5
LR_HEAD     = 1e-3
LR_FINETUNE = 1e-4
PATIENCE    = 3
AUTOTUNE    = tf.data.AUTOTUNE

mixed_precision.set_global_policy("mixed_float16")
strategy = tf.distribute.MirroredStrategy()
print(f"✔  Replicas in sync: {strategy.num_replicas_in_sync}")
GLOBAL_BATCH = BATCH_SIZE * strategy.num_replicas_in_sync
print(f"✔  Effective global batch size: {GLOBAL_BATCH}")

✔  Replicas in sync: 2
✔  Effective global batch size: 64


In [ ]:
# Path Sampling

VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def _collect_paths(directory, n_total, seed=42):
    paths = [
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if os.path.splitext(f)[1].lower() in VALID_EXT
    ]
    random.seed(seed)
    random.shuffle(paths)
    if len(paths) < n_total:
        raise ValueError(f"Only {len(paths)} images in {directory}, need {n_total}.")
    return paths[:n_total]


def get_split_paths(seed=42):
    """Returns (tr_paths, tr_labels, va_paths, va_labels, te_paths, te_labels).
    Labels: 0 = real, 1 = fake."""
    n_total    = N_TRAIN + N_VAL + N_TEST
    real_paths = _collect_paths(REAL_DIR, n_total, seed)
    fake_paths = _collect_paths(FAKE_DIR, n_total, seed)

    def split(paths, label):
        train = [(p, label) for p in paths[:N_TRAIN]]
        val   = [(p, label) for p in paths[N_TRAIN: N_TRAIN + N_VAL]]
        test  = [(p, label) for p in paths[N_TRAIN + N_VAL:]]
        return train, val, test

    r_tr, r_va, r_te = split(real_paths, 0)
    f_tr, f_va, f_te = split(fake_paths, 1)

    def unzip(lst):
        p, l = zip(*lst)
        return list(p), list(l)

    train_data = r_tr + f_tr; random.shuffle(train_data)
    val_data   = r_va + f_va; random.shuffle(val_data)
    test_data  = r_te + f_te; random.shuffle(test_data)

    return (*unzip(train_data), *unzip(val_data), *unzip(test_data))


print("⏳  Sampling dataset paths …")
tr_p, tr_l, va_p, va_l, te_p, te_l = get_split_paths()
print(f"✔  train={len(tr_p):,}  val={len(va_p):,}  test={len(te_p):,}")

⏳  Sampling dataset paths …
✔  train=10,000  val=2,000  test=2,000


In [ ]:
# TF Data Pipeline

def _decode_resize(path, label, img_size):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [img_size, img_size])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label


def _augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    return img, label


def make_dataset(paths, labels, img_size, training=False,
                 batch_size=GLOBAL_BATCH):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda p, l: _decode_resize(p, l, img_size),
                num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(_augment, num_parallel_calls=AUTOTUNE)
        ds = ds.shuffle(buffer_size=2_000, seed=42)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(AUTOTUNE)
    return ds

In [ ]:
# Model Builder (+ LoRA option)

_BACKBONE_MAP = {
    "B0": keras.applications.EfficientNetV2B0,
    "B3": keras.applications.EfficientNetV2B3,
}

def build_model(backbone_key="B0", img_size=224, freeze_backbone=True,
                unfreeze_last_n=0, extra_input_channels=0):
    """
    Build an EfficientNetV2 binary classifier inside MirroredStrategy scope.

    unfreeze_last_n:
        0   → head-only (backbone fully frozen)
        N   → last N backbone layers unfrozen
        -1  → full fine-tune (all layers trainable)

    extra_input_channels:
        0   → standard 3-channel RGB input
        3   → 6-channel RGB + FFT input (Phase 4)
    """
    n_channels  = 3 + extra_input_channels
    BackboneCls = _BACKBONE_MAP[backbone_key]

    with strategy.scope():
        inputs = keras.Input(shape=(img_size, img_size, n_channels), name="input")

        rgb = inputs[:, :, :, :3] if extra_input_channels > 0 else inputs

        backbone = BackboneCls(
            include_top=False,
            weights="imagenet",
            input_shape=(img_size, img_size, 3),
        )
        backbone.trainable = not freeze_backbone
        if freeze_backbone and unfreeze_last_n > 0:
            for layer in backbone.layers[-unfreeze_last_n:]:
                layer.trainable = True
        if unfreeze_last_n == -1:
            backbone.trainable = True

        x = backbone(rgb, training=False)

        if extra_input_channels > 0:
            fft_ch   = inputs[:, :, :, 3:]
            fft_feat = layers.GlobalAveragePooling2D()(
                           layers.Conv2D(32, 3, padding="same", activation="relu")(fft_ch))
            fft_feat = layers.Dense(128, activation="relu")(fft_feat)
            bb_feat  = layers.GlobalAveragePooling2D()(x)
            x        = layers.Concatenate()([bb_feat, fft_feat])
            x        = layers.Dense(256, activation="relu")(x)
            x        = layers.Dropout(0.4)(x)
        else:
            x = layers.GlobalAveragePooling2D()(x)
            x = layers.Dense(256, activation="relu")(x)
            x = layers.Dropout(0.4)(x)

        # Must be float32 under mixed precision
        outputs = layers.Dense(1, dtype="float32", name="logit")(x)

        model = keras.Model(inputs, outputs)
        model.compile(
            optimizer=keras.optimizers.Adam(LR_HEAD),
            loss=keras.losses.BinaryCrossentropy(from_logits=True),
            metrics=[
                keras.metrics.BinaryAccuracy(name="accuracy"),
                keras.metrics.AUC(name="auc"),
            ],
        )
    return model


def build_model_lora(
    backbone_key="B0",
    img_size=224,
    lora_rank=8,
    lora_alpha=16,
    lora_dropout=0.05,
):
    """
    LoRA fine-tuning: freeze backbone weights, inject low-rank adapters into Conv2D/Dense layers.

    Notes:
    - Requires Keras 3 (keras>=3.0). If you're on tf.keras only, this may not be available.
    - We keep the classifier head trainable as usual.
    """
    BackboneCls = _BACKBONE_MAP[backbone_key]

    with strategy.scope():
        inputs = keras.Input(shape=(img_size, img_size, 3), name="input")
        backbone = BackboneCls(
            include_top=False,
            weights="imagenet",
            input_shape=(img_size, img_size, 3),
        )
        backbone.trainable = False
        x = backbone(inputs, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(256, activation="relu")(x)
        x = layers.Dropout(0.4)(x)
        outputs = layers.Dense(1, dtype="float32", name="logit")(x)
        model = keras.Model(inputs, outputs)

        # Inject LoRA adapters after model is built
        if not hasattr(keras.layers, "LoRA"):
            raise RuntimeError(
                "keras.layers.LoRA not found. Install/upgrade to Keras 3 (keras>=3) "
                "or a TF/Keras build that includes LoRA.")

        # Wrap target layers with LoRA. Keep count small for speed.
        # EfficientNetV2 uses many Conv2D layers; LoRA there is typically effective.
        wrapped = 0
        for i, layer in enumerate(list(model.layers)):
            if isinstance(layer, layers.Conv2D) or isinstance(layer, layers.Dense):
                # Skip the final logit layer; it is already tiny and fully trainable
                if layer.name == "logit":
                    continue
                try:
                    model.layers[i] = keras.layers.LoRA(
                        layer,
                        rank=lora_rank,
                        alpha=lora_alpha,
                        dropout=lora_dropout,
                    )
                    wrapped += 1
                except Exception:
                    # Some layers may not be compatible; just skip them
                    pass

        model.compile(
            optimizer=keras.optimizers.Adam(LR_FINETUNE),
            loss=keras.losses.BinaryCrossentropy(from_logits=True),
            metrics=[
                keras.metrics.BinaryAccuracy(name="accuracy"),
                keras.metrics.AUC(name="auc"),
            ],
        )
        print(f"✔  LoRA wrapped layers: {wrapped}")
    return model

In [ ]:
# Training Helper (+ timing + param reporting)

import time

class TimeHistory(keras.callbacks.Callback):
    """Tracks wall-clock training time."""
    def on_train_begin(self, logs=None):
        self._start = time.time()
    def on_train_end(self, logs=None):
        self.elapsed_s = time.time() - self._start


def _count_trainable_params(model: keras.Model) -> int:
    return int(np.sum([np.prod(w.shape) for w in model.trainable_weights]))


def train_and_evaluate(model, train_ds, val_ds, test_ds, label, lr=LR_HEAD):
    model.optimizer.learning_rate = lr

    t_cb = TimeHistory()
    callbacks = [
        t_cb,
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=PATIENCE, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
    ]

    trainable_params = _count_trainable_params(model)
    total_params = model.count_params()
    print(f"{'='*60}\n  Training: {label}\n{'='*60}")
    print(f"Trainable params: {trainable_params:,} / Total params: {total_params:,}")

    history = model.fit(train_ds, validation_data=val_ds,
                        epochs=EPOCHS, callbacks=callbacks, verbose=1)

    test_loss, test_acc, test_auc = model.evaluate(test_ds, verbose=0)
    print(f"\n  ▶ [{label}]  loss={test_loss:.4f}  acc={test_acc:.4f}  auc={test_auc:.4f}")
    print(f"  ▶ [{label}]  train_time={t_cb.elapsed_s:.1f}s")

    return dict(
        label=label,
        history=history.history,
        test_loss=test_loss,
        test_acc=test_acc,
        test_auc=test_auc,
        train_time_s=t_cb.elapsed_s,
        trainable_params=trainable_params,
        total_params=total_params,
    )

In [ ]:
# PHASE 1 — Capacity Comparison: EfficientNetV2-B0 vs B3 (head-only)

phase1_results = []

for backbone, img_size in [("B0", 224), ("B3", 300)]:
    tr_ds = make_dataset(tr_p, tr_l, img_size, training=True)
    va_ds = make_dataset(va_p, va_l, img_size)
    te_ds = make_dataset(te_p, te_l, img_size)

    model = build_model(backbone_key=backbone, img_size=img_size, freeze_backbone=True)

    res = train_and_evaluate(
        model, tr_ds, va_ds, te_ds,
        label=f"Phase1_EfficientNetV2-{backbone}_{img_size}x{img_size}",
        lr=LR_HEAD,
    )
    phase1_results.append(res)

print(f"\n{'Experiment':<46} {'Test Acc':>9}  {'Test AUC':>9}  {'Test Loss':>10}")
print("-" * 74)
for r in phase1_results:
    print(f"{r['label']:<46} {r['test_acc']:>9.4f}  {r['test_auc']:>9.4f}  {r['test_loss']:>10.4f}")

  Training: Phase1_EfficientNetV2-B0_224x224
Epoch 1/5


I0000 00:00:1772589535.083223    2691 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1772589535.154169    2689 cuda_dnn.cc:529] Loaded cuDNN version 91002


156/156 ━━━━━━━━━━━━━━━━━━━━ 45s 125ms/step - accuracy: 0.6940 - auc: 0.7491 - loss: 0.5552 - val_accuracy: 0.8241 - val_auc: 0.8622 - val_loss: 0.4192 - learning_rate: 0.0010
Epoch 2/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 91ms/step - accuracy: 0.7959 - auc: 0.8414 - loss: 0.4217 - val_accuracy: 0.8241 - val_auc: 0.8637 - val_loss: 0.4438 - learning_rate: 0.0010
Epoch 3/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 17s 96ms/step - accuracy: 0.8172 - auc: 0.8650 - loss: 0.3868 - val_accuracy: 0.8261 - val_auc: 0.8614 - val_loss: 0.4406 - learning_rate: 0.0010
Epoch 4/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 90ms/step - accuracy: 0.8391 - auc: 0.8797 - loss: 0.3462 - val_accuracy: 0.8463 - val_auc: 0.8875 - val_loss: 0.3475 - learning_rate: 0.0010
Epoch 5/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 17s 92ms/step - accuracy: 0.8507 - auc: 0.8855 - loss: 0.3304 - val_accuracy: 0.8574 - val_auc: 0.8932 - val_loss: 0.3376 - learning_rate: 0.0010

  ▶ [Phase1_EfficientNetV2-B0_224x224]  loss=0.3273  acc=0.8715  auc=0.8964
  Train

In [ ]:
# PHASE 2 — Fine-tuning Depth: head-only / partial / full (B0, 224×224)

IMG_SIZE_P2 = 224
tr_ds_p2 = make_dataset(tr_p, tr_l, IMG_SIZE_P2, training=True)
va_ds_p2 = make_dataset(va_p, va_l, IMG_SIZE_P2)
te_ds_p2 = make_dataset(te_p, te_l, IMG_SIZE_P2)

phase2_configs = [
    {"label": "Phase2_HeadOnly",               "freeze": True,  "unfreeze_last": 0,  "lr": LR_HEAD},
    {"label": "Phase2_PartialUnfreeze_last30",  "freeze": True,  "unfreeze_last": 30, "lr": LR_FINETUNE},
    {"label": "Phase2_FullFineTune",            "freeze": False, "unfreeze_last": -1, "lr": LR_FINETUNE},
]

phase2_results = []

for cfg in phase2_configs:
    model = build_model(
        backbone_key="B0",
        img_size=IMG_SIZE_P2,
        freeze_backbone=cfg["freeze"],
        unfreeze_last_n=cfg["unfreeze_last"],
    )
    res = train_and_evaluate(
        model, tr_ds_p2, va_ds_p2, te_ds_p2,
        label=cfg["label"],
        lr=cfg["lr"],
    )
    phase2_results.append(res)

print(f"\n{'Experiment':<46} {'Test Acc':>9}  {'Test AUC':>9}  {'Test Loss':>10}")
print("-" * 74)
for r in phase2_results:
    print(f"{r['label']:<46} {r['test_acc']:>9.4f}  {r['test_auc']:>9.4f}  {r['test_loss']:>10.4f}")

  Training: Phase2_HeadOnly
Epoch 1/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 33s 123ms/step - accuracy: 0.6889 - auc: 0.7425 - loss: 0.5584 - val_accuracy: 0.8125 - val_auc: 0.8574 - val_loss: 0.4161 - learning_rate: 0.0010
Epoch 2/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 91ms/step - accuracy: 0.8006 - auc: 0.8403 - loss: 0.4232 - val_accuracy: 0.8306 - val_auc: 0.8773 - val_loss: 0.4029 - learning_rate: 0.0010
Epoch 3/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 89ms/step - accuracy: 0.8227 - auc: 0.8608 - loss: 0.3843 - val_accuracy: 0.8211 - val_auc: 0.8540 - val_loss: 0.4605 - learning_rate: 0.0010
Epoch 4/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 15s 84ms/step - accuracy: 0.8361 - auc: 0.8784 - loss: 0.3491 - val_accuracy: 0.8448 - val_auc: 0.8833 - val_loss: 0.3479 - learning_rate: 0.0010
Epoch 5/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 17s 94ms/step - accuracy: 0.8426 - auc: 0.8815 - loss: 0.3306 - val_accuracy: 0.8548 - val_auc: 0.8849 - val_loss: 0.3662 - learning_rate: 0.0010

  ▶ [Phase2_HeadOnly]  loss=0.3505  acc=0.861

In [ ]:
# PHASE 3 — Input Resolution Ablation: 224 / 300 / 384 (B0, head-only)

phase3_results = []

for img_size in [224, 300, 384]:
    tr_ds = make_dataset(tr_p, tr_l, img_size, training=True)
    va_ds = make_dataset(va_p, va_l, img_size)
    te_ds = make_dataset(te_p, te_l, img_size)

    model = build_model(backbone_key="B0", img_size=img_size, freeze_backbone=True)

    res = train_and_evaluate(
        model, tr_ds, va_ds, te_ds,
        label=f"Phase3_B0_{img_size}x{img_size}",
        lr=LR_HEAD,
    )
    phase3_results.append(res)

print(f"\n{'Experiment':<46} {'Test Acc':>9}  {'Test AUC':>9}  {'Test Loss':>10}")
print("-" * 74)
for r in phase3_results:
    print(f"{r['label']:<46} {r['test_acc']:>9.4f}  {r['test_auc']:>9.4f}  {r['test_loss']:>10.4f}")

  Training: Phase3_B0_224x224
Epoch 1/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 33s 124ms/step - accuracy: 0.6998 - auc: 0.7493 - loss: 0.5499 - val_accuracy: 0.8261 - val_auc: 0.8587 - val_loss: 0.4325 - learning_rate: 0.0010
Epoch 2/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 91ms/step - accuracy: 0.8079 - auc: 0.8456 - loss: 0.4148 - val_accuracy: 0.8165 - val_auc: 0.8621 - val_loss: 0.4484 - learning_rate: 0.0010
Epoch 3/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 89ms/step - accuracy: 0.8210 - auc: 0.8605 - loss: 0.3913 - val_accuracy: 0.8306 - val_auc: 0.8694 - val_loss: 0.4264 - learning_rate: 0.0010
Epoch 4/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 15s 82ms/step - accuracy: 0.8365 - auc: 0.8716 - loss: 0.3589 - val_accuracy: 0.8523 - val_auc: 0.8854 - val_loss: 0.3543 - learning_rate: 0.0010
Epoch 5/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 90ms/step - accuracy: 0.8496 - auc: 0.8885 - loss: 0.3257 - val_accuracy: 0.8508 - val_auc: 0.8918 - val_loss: 0.3620 - learning_rate: 0.0010

  ▶ [Phase3_B0_224x224]  loss=0.3459  acc=0

In [ ]:
# FFT Pipeline Helpers

def _compute_fft_magnitude(img):
    """img: [H,W,3] float32 (0-255). Returns [H,W,3] log-magnitude FFT scaled 0-255."""
    channels = tf.unstack(img / 255.0, axis=-1)
    fft_mags = []
    for ch in channels:
        fft     = tf.signal.fft2d(tf.cast(ch, tf.complex64))
        mag     = tf.abs(tf.signal.fftshift(fft))
        log_mag = tf.math.log1p(mag)
        log_mag = log_mag / (tf.reduce_max(log_mag) + 1e-8) * 255.0
        fft_mags.append(log_mag)
    return tf.stack(fft_mags, axis=-1)  # [H, W, 3]


def _decode_resize_fft(path, label, img_size):
    """Returns [H,W,6]: first 3 channels = RGB, last 3 = FFT magnitude."""
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [img_size, img_size])
    img = tf.cast(img, tf.float32)
    fft = _compute_fft_magnitude(img)
    return tf.concat([img, fft], axis=-1), label


def _augment_6ch(img6, label):
    """Augment only RGB channels; leave FFT channels unchanged."""
    rgb = tf.image.random_flip_left_right(img6[:, :, :3])
    rgb = tf.image.random_brightness(rgb, 0.2)
    rgb = tf.image.random_contrast(rgb, 0.8, 1.2)
    return tf.concat([rgb, img6[:, :, 3:]], axis=-1), label


def make_fft_dataset(paths, labels, img_size, training=False,
                     batch_size=GLOBAL_BATCH):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda p, l: _decode_resize_fft(p, l, img_size),
                num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(_augment_6ch, num_parallel_calls=AUTOTUNE)
        ds = ds.shuffle(buffer_size=2_000, seed=42)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(AUTOTUNE)
    return ds

In [ ]:
# PHASE 4 — Frequency-Domain Features: RGB + FFT Magnitude (B0, 224, head-only)

IMG_SIZE_P4 = 224
phase4_results = []

tr_rgb = make_dataset(tr_p, tr_l, IMG_SIZE_P4, training=True)
va_rgb = make_dataset(va_p, va_l, IMG_SIZE_P4)
te_rgb = make_dataset(te_p, te_l, IMG_SIZE_P4)

model_rgb = build_model(backbone_key="B0", img_size=IMG_SIZE_P4,
                        freeze_backbone=True, extra_input_channels=0)
res_rgb = train_and_evaluate(model_rgb, tr_rgb, va_rgb, te_rgb,
                             label="Phase4_B0_RGBOnly_224", lr=LR_HEAD)
phase4_results.append(res_rgb)

tr_fft = make_fft_dataset(tr_p, tr_l, IMG_SIZE_P4, training=True)
va_fft = make_fft_dataset(va_p, va_l, IMG_SIZE_P4)
te_fft = make_fft_dataset(te_p, te_l, IMG_SIZE_P4)

model_fft = build_model(backbone_key="B0", img_size=IMG_SIZE_P4,
                        freeze_backbone=True, extra_input_channels=3)
res_fft = train_and_evaluate(model_fft, tr_fft, va_fft, te_fft,
                             label="Phase4_B0_RGB+FFT_224", lr=LR_HEAD)
phase4_results.append(res_fft)

print(f"\n{'Experiment':<46} {'Test Acc':>9}  {'Test AUC':>9}  {'Test Loss':>10}")
print("-" * 74)
for r in phase4_results:
    print(f"{r['label']:<46} {r['test_acc']:>9.4f}  {r['test_auc']:>9.4f}  {r['test_loss']:>10.4f}")

  Training: Phase4_B0_RGBOnly_224
Epoch 1/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 35s 130ms/step - accuracy: 0.6950 - auc: 0.7475 - loss: 0.5558 - val_accuracy: 0.8216 - val_auc: 0.8656 - val_loss: 0.4209 - learning_rate: 0.0010
Epoch 2/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 17s 92ms/step - accuracy: 0.8020 - auc: 0.8429 - loss: 0.4145 - val_accuracy: 0.8085 - val_auc: 0.8523 - val_loss: 0.4746 - learning_rate: 0.0010
Epoch 3/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 17s 93ms/step - accuracy: 0.8180 - auc: 0.8583 - loss: 0.3883 - val_accuracy: 0.8251 - val_auc: 0.8660 - val_loss: 0.4261 - learning_rate: 0.0010
Epoch 4/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 15s 84ms/step - accuracy: 0.8357 - auc: 0.8745 - loss: 0.3501 - val_accuracy: 0.8528 - val_auc: 0.8864 - val_loss: 0.3479 - learning_rate: 0.0010
Epoch 5/5
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 86ms/step - accuracy: 0.8510 - auc: 0.8884 - loss: 0.3248 - val_accuracy: 0.8558 - val_auc: 0.8869 - val_loss: 0.3536 - learning_rate: 0.0010

  ▶ [Phase4_B0_RGBOnly_224]  loss=0.355

In [ ]:
# PHASE 5 — LoRA vs Full Fine-tune (B0, 224×224)

# Reuse the same split paths; build fresh datasets for fairness
IMG_SIZE_P5 = 224
tr_ds_p5 = make_dataset(tr_p, tr_l, IMG_SIZE_P5, training=True)
va_ds_p5 = make_dataset(va_p, va_l, IMG_SIZE_P5)
te_ds_p5 = make_dataset(te_p, te_l, IMG_SIZE_P5)

phase5_results = []

# Full fine-tune (baseline)
model_full_p5 = build_model(
    backbone_key="B0",
    img_size=IMG_SIZE_P5,
    freeze_backbone=False,
    unfreeze_last_n=-1,
    extra_input_channels=0,
 )
res_full_p5 = train_and_evaluate(
    model_full_p5, tr_ds_p5, va_ds_p5, te_ds_p5,
    label="Phase5_FullFineTune_B0_224",
    lr=LR_FINETUNE,
 )
phase5_results.append(res_full_p5)

# LoRA fine-tune (frozen backbone + adapter params)
try:
    model_lora_p5 = build_model_lora(
        backbone_key="B0",
        img_size=IMG_SIZE_P5,
        lora_rank=8,
        lora_alpha=16,
        lora_dropout=0.05,
    )
    res_lora_p5 = train_and_evaluate(
        model_lora_p5, tr_ds_p5, va_ds_p5, te_ds_p5,
        label="Phase5_LoRA_r8_a16_B0_224",
        lr=LR_FINETUNE,
    )
    phase5_results.append(res_lora_p5)
except RuntimeError as e:
    print("⚠️  LoRA not available in this environment; skipping LoRA run.")
    print("   Details:", e)

print(
    f"\n{'Experiment':<34} {'Test Acc':>9}  {'Test AUC':>9}  {'Time(s)':>9}  {'Trainable':>11}"
)
print("-" * 84)
for r in phase5_results:
    print(
        f"{r['label']:<34} "
        f"{r['test_acc']:>9.4f}  {r['test_auc']:>9.4f}  "
        f"{r['train_time_s']:>9.1f}  {r['trainable_params']:>11,}"
    )

# Quick derived comparison (only if LoRA ran)
if len(phase5_results) == 2:
    full, lora = phase5_results[0], phase5_results[1]
    speedup = full['train_time_s'] / max(lora['train_time_s'], 1e-9)
    dacc = lora['test_acc'] - full['test_acc']
    print("\nSummary:")
    print(f"- LoRA speedup: {speedup:.2f}× faster (wall time)")
    print(f"- Accuracy delta (LoRA - Full): {dacc:+.4f}")